# UPET / PET-MAD foundation model · GMD-26 force-error gallery (zero-shot)

Third notebook in the series, same five GMD-26 *Fragment Duplication* trajectories and the
same two-sphere force-error encoding as `simurgh_gmd_force_error_gallery.ipynb` (trained,
task-specific) and `mace_foundation_gmd_gallery.ipynb` (universal, MPTrj). Here the
potential is **PET-MAD**, loaded through `goal.md`'s `CalculatorFactory`.

**UPET *is* PET-MAD.** The `pet-mad` package was renamed to `upet` upstream
(`lab-cosmo/upet`) to cover the wider family — `pet-mad`, `pet-omat`, `pet-oam`,
`pet-omad`, `pet-omatpes`, `pet-spice`. `pet-mad` 1.4.4 is the final release under the old
name. GOAL's `key="upet"` drives the new package.

**What is being compared.** PET-MAD-1.5 is trained on the MAD dataset at **r2SCAN**;
GMD's references are molecular **PBE/def2-SVP**. So this measures transferability of a
universal model to a functional it was never fit to — forces are the meaningful column,
energies only after the per-element baseline of §3.5. §6 also runs **PET-SPICE**, the
molecular-dataset member of the same family, to test the obvious hypothesis that a
molecule-trained foundation model should do better on carboxylic acids.

**Kernel** — `GOAL (mace-cuda, standalone)` (`.venv-mace`), the same one the MACE notebook
uses; `upet` was installed alongside it.

## 0 · Environment shims

Two workarounds that must run before anything else. Both are environment quirks, not GOAL
or UPET problems, and both are safe to delete once the underlying causes are gone.

In [ ]:
import socket

import torch

# (1) e3nn 0.4.4 (pinned by mace-torch, which shares this venv) `torch.load`s a constants
#     file holding a `slice`, refused by torch >= 2.6 under weights_only=True.
torch.serialization.add_safe_globals([slice])

# (2) huggingface.co resolves to IPv6 first on this host, and IPv6 egress is blackholed.
#     curl recovers in milliseconds; httpx (used by huggingface_hub) does not fail over, so
#     UPET's model resolution stalls for minutes. Sorting IPv4 addresses first fixes it
#     without disabling IPv6 — measured: 80 s -> 0.16 s per HF call, model load 10 min -> 1 s.
_orig_getaddrinfo = socket.getaddrinfo


def _ipv4_first(host, port, family=0, type=0, proto=0, flags=0):  # noqa: A002
    infos = _orig_getaddrinfo(host, port, family, type, proto, flags)
    return sorted(infos, key=lambda info: 0 if info[0] == socket.AF_INET else 1)


socket.getaddrinfo = _ipv4_first

print(f"torch {torch.__version__} · CUDA available: {torch.cuda.is_available()}")

In [ ]:
from __future__ import annotations

import io
import time
from pathlib import Path

import ase
import ase.io
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from ase.neighborlist import NeighborList, natural_cutoffs
from matplotlib.cm import ScalarMappable

import ipywidgets as widgets
import py3Dmol
from IPython.display import HTML, display

import upet
from goal.md.core.calculator_factory import CalculatorFactory

print(f"upet {upet.__version__ if hasattr(upet, '__version__') else '0.2.x'}")

## 1 · Configuration

`version` is pinned rather than left at `"latest"`: UPET resolves `latest` against the
HuggingFace repo at call time, so an unpinned notebook silently changes weights the day
upstream publishes a new release. (GOAL's builder only started forwarding `version` when
this notebook was written — see the §7 note on the builder fixes.)

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication"

MODEL, VERSION = "pet-mad-s", "1.5.0"  # universal, MAD dataset, r2SCAN
COMPARE_MODEL, COMPARE_VERSION = "pet-spice-s", "0.2.0"  # same family, molecular dataset

TRAJECTORIES: list[tuple[str, str]] = [
    ("TestID", "O2C4H8_PBE.traj"),
    ("TestID", "O2C10H20_PBE.traj"),
    ("TestOOD", "O4C4H6_PBE.traj"),
    ("TestOOD", "O4C7H12_PBE.traj"),
    ("TestOOD", "O4C10H18_PBE.traj"),
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_STRIDE = 1  # >1 subsamples frames (each trajectory holds 200)
PLAYBACK_FPS = 3

assert DATA_ROOT.is_dir(), f"GMD data not found: {DATA_ROOT}"
print(f"data   : {DATA_ROOT}")
print(f"model  : {MODEL} v{VERSION}   (comparison: {COMPARE_MODEL} v{COMPARE_VERSION})")
print(f"device : {DEVICE}")

## 2 · Load PET-MAD through GOAL's calculator factory

`CalculatorFactory.create("upet", …)` validates the model id against
`upet._version.UPET_AVAILABLE_MODELS` and returns upstream's `UPETCalculator`, an ordinary
ASE calculator. This is the same call `configs/md/calculators/upet.yaml` makes, so
everything under `goal.md` — MTS/RESPA, i-PI, online learning — takes this model by
editing one YAML key.

**Interface asymmetry worth knowing.** MACE and UMA have ml-side GOAL backbones
(`goal.ml.nn.models.foundation.{mace,uma}`) that consume `AtomicGraph`, so the MACE
notebook batches frames through `module(batch)`. UPET has no such backbone: metatensor
`TensorMap` systems would need their own adapter (tracked as future work in
`goal.ml.cli.finetune_upet`), and fine-tuning goes through metatrain as a subprocess via
`goal-finetune-upet`. So this notebook predicts **one frame at a time** through ASE. At
~20 ms/frame that costs a few seconds per trajectory — not worth an adapter yet.

First call contacts HuggingFace and caches under `~/.cache/huggingface/`; later runs are
offline-ish (UPET still lists the repo tree, hence shim (2) above).

In [ ]:
t0 = time.perf_counter()
calc = CalculatorFactory.create("upet", model=MODEL, version=VERSION, device=DEVICE)
print(f"loaded in {time.perf_counter() - t0:.1f}s")
print(f"calculator : {type(calc).__name__}")
print(f"properties : {sorted(calc.implemented_properties)}")

## 3 · Per-frame inference

In [ ]:
PREDICTIONS: dict[tuple[str, str, str], dict[str, object]] = {}


def predict_frames(frames: list[ase.Atoms], calculator) -> tuple[np.ndarray, list[np.ndarray]]:
    """Energy + forces for every frame, one ASE single point at a time."""
    energies, forces = [], []
    for frame in frames:
        atoms = frame.copy()
        atoms.calc = calculator
        energies.append(atoms.get_potential_energy())
        forces.append(atoms.get_forces())
    return np.array(energies), forces


def evaluate_trajectory(
    split: str,
    molecule: str,
    calculator=None,
    tag: str | None = None,
    stride: int = FRAME_STRIDE,
) -> dict[str, object]:
    """Load one trajectory, predict every frame, derive the per-atom error fields."""
    calculator = calc if calculator is None else calculator
    tag = tag or MODEL
    key = (tag, split, molecule)
    if key in PREDICTIONS:
        return PREDICTIONS[key]

    path = DATA_ROOT / split / molecule
    if not path.is_file():
        available = sorted(p.name for p in (DATA_ROOT / split).glob("*.traj"))
        raise FileNotFoundError(f"{path} not found. Available: {available}")

    frames = ase.io.read(str(path), index=f"::{stride}")
    ref_energy = np.array([f.get_potential_energy() for f in frames])
    ref_forces = [f.get_forces().copy() for f in frames]

    t_start = time.perf_counter()
    pred_energy, pred_forces = predict_frames(frames, calculator)
    elapsed = time.perf_counter() - t_start

    n_frames, n_atoms = len(frames), len(frames[0])
    rel_mag = np.zeros((n_frames, n_atoms))
    cos_sim = np.zeros((n_frames, n_atoms))
    for i, (pred, ref) in enumerate(zip(pred_forces, ref_forces)):
        norm_pred = np.linalg.norm(pred, axis=1)
        norm_ref = np.linalg.norm(ref, axis=1)
        denom = norm_pred * norm_ref
        cos_sim[i] = np.divide(
            (pred * ref).sum(axis=1), denom, out=np.zeros_like(denom), where=denom > 1e-20
        )
        rel_mag[i] = np.abs(norm_pred - norm_ref) / (norm_ref + 1e-10)

    record: dict[str, object] = {
        "tag": tag,
        "split": split,
        "molecule": molecule,
        "formula": frames[0].get_chemical_formula(),
        "symbols": frames[0].get_chemical_symbols(),
        "numbers": frames[0].get_atomic_numbers(),
        "n_frames": n_frames,
        "n_atoms": n_atoms,
        "positions": np.array([f.get_positions() for f in frames]),
        "ref_energy": ref_energy,
        "pred_energy": pred_energy,
        "ref_forces": ref_forces,
        "pred_forces": pred_forces,
        "rel_mag": rel_mag,
        "cos_sim": cos_sim,
        "forces_mae": float(np.mean([np.abs(p - r).mean() for p, r in zip(pred_forces, ref_forces)])),
        "seconds": elapsed,
    }
    PREDICTIONS[key] = record
    print(
        f"[{tag}] {split}/{molecule}: {n_frames} frames × {n_atoms} atoms in {elapsed:.1f}s "
        f"({elapsed / n_frames * 1e3:.0f} ms/frame)"
    )
    return record

## 4 · Sanity check — one frame against DFT

`‖ΣF‖` is the Newton check: a conservative model derives forces from `-∂E/∂r`, so they must
sum to zero. Non-zero here would mean `non_conservative=True` slipped in (the direct force
head), which `goal.md` refuses by default and warns loudly about.

In [ ]:
probe = ase.io.read(str(DATA_ROOT / "TestID" / "O2C6H12_PBE.traj"), index=100)
probe_ref_e = probe.get_potential_energy()
probe_ref_f = probe.get_forces().copy()
probe_pred_e, (probe_pred_f,) = predict_frames([probe], calc)

norm_pred = np.linalg.norm(probe_pred_f, axis=1)
norm_ref = np.linalg.norm(probe_ref_f, axis=1)
probe_cos = (probe_pred_f * probe_ref_f).sum(axis=1) / (norm_pred * norm_ref + 1e-12)

print(f"{probe.get_chemical_formula()} · frame 100 · {len(probe)} atoms · {MODEL} v{VERSION}\n")
print(f"  {'energy (eV)':24s} DFT {probe_ref_e:14.4f}   PET-MAD {probe_pred_e[0]:12.4f}")
print(f"  {'raw ΔE (eV)':24s}     {probe_pred_e[0] - probe_ref_e:14.4f}"
      f"   ← r2SCAN vs PBE/def2-SVP, see §5")
print(f"  {'forces MAE (eV/Å)':24s}     {np.abs(probe_pred_f - probe_ref_f).mean():14.5f}")
print(f"  {'mean cos(F_pred,F_ref)':24s}     {probe_cos.mean():14.5f}")
print(f"  {'net force ‖ΣF‖ (eV/Å)':24s} DFT {np.linalg.norm(probe_ref_f.sum(axis=0)):14.2e}"
      f"   PET-MAD {np.linalg.norm(probe_pred_f.sum(axis=0)):9.2e}")

## 5 · Energies need a per-element baseline

Same treatment as the MACE notebook, and for the same reason: a foundation model's total
energies live on its own training reference (here r2SCAN), so the gap to GMD's PBE/def2-SVP
is dominated by a sum of per-element constants. Fit
`E_DFT − E_pred ≈ Σ_Z n_Z · Δe_Z` by least squares over every frame of all five
trajectories (3 unknowns for H, C, O), then report the residual. Fitted on the same frames
it is scored on — a 3-parameter alignment, not a held-out claim. Forces need none of this.

In [ ]:
def fit_element_baseline(records: list[dict]) -> dict[int, float]:
    """Least-squares per-element offsets Δe_Z from  E_DFT − E_pred ≈ Σ_Z n_Z Δe_Z."""
    elements = sorted({int(z) for r in records for z in set(r["numbers"])})
    design, residual = [], []
    for r in records:
        counts = np.array([[np.sum(np.asarray(r["numbers"]) == z) for z in elements]], dtype=float)
        design.append(np.repeat(counts, r["n_frames"], axis=0))
        residual.append(np.asarray(r["ref_energy"]) - np.asarray(r["pred_energy"]))
    offsets, *_ = np.linalg.lstsq(np.vstack(design), np.concatenate(residual), rcond=None)
    return {z: float(o) for z, o in zip(elements, offsets)}


def aligned_energy_mae_per_atom(record: dict, baseline: dict[int, float]) -> float:
    shift = float(sum(baseline[int(z)] for z in np.asarray(record["numbers"])))
    delta = np.asarray(record["pred_energy"]) + shift - np.asarray(record["ref_energy"])
    return float(np.abs(delta).mean() / record["n_atoms"])


print(f"evaluating all five trajectories with {MODEL} (cached for §7 and §8) ...")
rows = [evaluate_trajectory(split, molecule) for split, molecule in TRAJECTORIES]
BASELINE = fit_element_baseline(rows)

print("\nfitted per-element offsets  Δe_Z  (eV/atom):")
for z, offset in BASELINE.items():
    print(f"  Z={z:2d}  {offset:+12.4f}")
print("\nper-trajectory energy error, raw vs baseline-aligned:")
print(f"  {'molecule':10s} {'raw |ΔE|/atom':>14s} {'aligned |ΔE|/atom':>18s}")
for r in rows:
    raw = np.abs(np.asarray(r["pred_energy"]) - np.asarray(r["ref_energy"])).mean() / r["n_atoms"]
    r["energy_mae_per_atom"] = aligned_energy_mae_per_atom(r, BASELINE)
    print(f"  {r['formula']:10s} {raw:14.3f} {r['energy_mae_per_atom']:18.5f}")

## 6 · Viewer

Identical to the other two galleries, so the three notebooks are visually comparable. The
magnitude colour scale is per-trajectory (1st–99th percentile), so read the colourbar, not
the colour, when comparing across notebooks.

In [ ]:
CMAP_MAG = mcolors.LinearSegmentedColormap.from_list("mag_err", ["#ffffff", "#ffb6d5", "#ff1493"])
CMAP_DIR = mcolors.LinearSegmentedColormap.from_list(
    "dir_err", [(0.0, "#ffffff"), (0.5, "#e60026"), (1.0, "#ffd000")]
)
CMAP_DIR_CB = mcolors.LinearSegmentedColormap.from_list(
    "dir_err_cb", [(0.0, "#ffd000"), (0.5, "#e60026"), (1.0, "#ffffff")]
)
OUTER_OPACITY_MIN, OUTER_OPACITY_MAX = 0.25, 0.85
VIEWER_WIDTH, VIEWER_HEIGHT = 600, 460


def kabsch_align(mobile: np.ndarray, ref: np.ndarray) -> np.ndarray:
    """Rigidly rotate + translate `mobile` (n_atoms, 3) onto `ref`, minimising RMSD."""
    mobile_com = mobile.mean(axis=0)
    ref_com = ref.mean(axis=0)
    H = (mobile - mobile_com).T @ (ref - ref_com)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return (R @ (mobile - mobile_com).T).T + ref_com


def colorbar_figure(norm, cmap, label):
    fig, ax = plt.subplots(figsize=(1.0, 3.6))
    fig.subplots_adjust(left=0.45, right=0.6, top=0.92, bottom=0.1)
    fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=ax, orientation="vertical").set_label(
        label, fontsize=8
    )
    plt.close(fig)
    return fig


def show_trajectory(split: str, molecule: str) -> None:
    """Render the interactive force-error playback for one GMD trajectory."""
    record = evaluate_trajectory(split, molecule)
    symbols: list[str] = record["symbols"]  # type: ignore[assignment]
    n_atoms: int = record["n_atoms"]  # type: ignore[assignment]
    n_frames: int = record["n_frames"]  # type: ignore[assignment]
    rel_mag: np.ndarray = record["rel_mag"]  # type: ignore[assignment]
    cos_sim: np.ndarray = record["cos_sim"]  # type: ignore[assignment]

    positions = np.array(record["positions"])
    reference = positions[0].copy()
    positions = np.array([kabsch_align(p, reference) for p in positions])

    mag_vmin, mag_vmax = np.percentile(rel_mag, [1, 99])
    if mag_vmax <= mag_vmin:
        mag_vmax = mag_vmin + 1e-6

    # Fluctuation about this trajectory's own mean — the §5 constant would swamp it.
    d_energy = np.asarray(record["pred_energy"]) - np.asarray(record["ref_energy"])
    d_energy = d_energy - d_energy.mean()

    def two_sphere_colors(frame_mag: np.ndarray, frame_cos: np.ndarray):
        inner = [mcolors.to_hex(CMAP_MAG(min(float(m) / mag_vmax, 1.0))) for m in frame_mag]
        t_dir = (1.0 - frame_cos) / 2.0
        outer = [mcolors.to_hex(CMAP_DIR(t)) for t in t_dir]
        opacity = OUTER_OPACITY_MIN + t_dir * (OUTER_OPACITY_MAX - OUTER_OPACITY_MIN)
        return inner, outer, opacity

    first_frame = ase.Atoms(symbols=symbols, positions=positions[0])
    nl = NeighborList(natural_cutoffs(first_frame, mult=1.1), self_interaction=False, bothways=False)
    nl.update(first_frame)
    bonds = [(i, j) for i in range(n_atoms) for j in nl.get_neighbors(i)[0] if j > i]

    oxygen_idx = [i for i, s in enumerate(symbols) if s == "O"]
    first_carbon_idx = next((i for i, s in enumerate(symbols) if s == "C"), None)
    tracked_labels = {idx: f"O{n + 1}" for n, idx in enumerate(oxygen_idx)}
    if first_carbon_idx is not None:
        tracked_labels[first_carbon_idx] = "C1"

    def build_view(frame_idx: int) -> py3Dmol.view:
        frame_pos = positions[frame_idx]
        inner, outer, opacity = two_sphere_colors(rel_mag[frame_idx], cos_sim[frame_idx])

        buf = io.StringIO()
        ase.io.write(buf, ase.Atoms(symbols=symbols, positions=frame_pos), format="xyz")

        view = py3Dmol.view(width=VIEWER_WIDTH, height=VIEWER_HEIGHT)
        view.addModel(buf.getvalue(), "xyz")
        view.setStyle({}, {})  # suppress py3Dmol's own bonds; we draw them below

        for i, j in bonds:
            p1, p2 = frame_pos[i], frame_pos[j]
            view.addCylinder({
                "start": {"x": float(p1[0]), "y": float(p1[1]), "z": float(p1[2])},
                "end": {"x": float(p2[0]), "y": float(p2[1]), "z": float(p2[2])},
                "radius": 0.08,
                "color": "#888888",
                "fromCap": True,
                "toCap": True,
            })

        for i, pos in enumerate(frame_pos):
            x, y, z = float(pos[0]), float(pos[1]), float(pos[2])
            view.addSphere({"center": {"x": x, "y": y, "z": z}, "radius": 0.42,
                            "color": outer[i], "opacity": float(opacity[i])})
            view.addSphere({"center": {"x": x, "y": y, "z": z}, "radius": 0.22,
                            "color": inner[i]})

        for i, (sym, pos) in enumerate(zip(symbols, frame_pos)):
            view.addLabel(tracked_labels.get(i, sym), {
                "position": {"x": float(pos[0]), "y": float(pos[1]), "z": float(pos[2])},
                "fontSize": 12 if i in tracked_labels else 11,
                "fontColor": "black",
                "showBackground": False,
                "alignment": "center",
            })

        view.zoomTo()
        return view

    viewer_out, info_out, err_out = widgets.Output(), widgets.Output(), widgets.Output()
    cb_mag_out, cb_dir_out = widgets.Output(), widgets.Output()
    with cb_mag_out:
        display(colorbar_figure(mcolors.Normalize(0.0, float(mag_vmax)), CMAP_MAG,
                                "Inner — relative |F| error"))
    with cb_dir_out:
        display(colorbar_figure(mcolors.Normalize(-1.0, 1.0), CMAP_DIR_CB,
                                "Outer — cosine similarity"))

    err_fig, err_ax = plt.subplots(figsize=(9.5, 2.0))
    err_ax.plot(rel_mag.mean(axis=1), color="darkmagenta")
    err_ax.set_xlim(0, n_frames - 1)
    err_ax.set_xlabel("frame", fontsize=9)
    err_ax.set_ylabel("|F| error", color="darkmagenta", fontsize=9)
    err_ax.tick_params(axis="y", labelcolor="darkmagenta", labelsize=8)
    err_ax.tick_params(axis="x", labelsize=8)
    err_ax2 = err_ax.twinx()
    err_ax2.plot(cos_sim.mean(axis=1), color="firebrick", alpha=0.7)
    err_ax2.set_ylabel("cos sim", color="firebrick", fontsize=9)
    err_ax2.tick_params(axis="y", labelcolor="firebrick", labelsize=8)
    err_ax2.set_ylim(-1.05, 1.05)
    err_marker = err_ax.axvline(0, color="black", linewidth=1.2)
    err_fig.tight_layout()
    plt.close(err_fig)

    def render_frame(frame_idx: int) -> None:
        view = build_view(frame_idx)
        viewer_out.clear_output(wait=True)
        with viewer_out:
            view.show()

        err_marker.set_xdata([frame_idx, frame_idx])
        err_out.clear_output(wait=True)
        with err_out:
            display(err_fig)

        mag, cos = rel_mag[frame_idx], cos_sim[frame_idx]
        info_out.clear_output(wait=True)
        with info_out:
            display(HTML(
                f"<div style='font-family:sans-serif;font-size:13px;line-height:1.7;padding-left:8px'>"
                f"<b>{record['formula']}<br>{split} · {Path(molecule).stem}</b><br>"
                f"model = {MODEL} v{VERSION}<br>"
                f"zero-shot (r2SCAN)<br><br>"
                f"frame = {frame_idx}/{n_frames - 1}<br>"
                f"mean |F| err&nbsp;= {mag.mean():.4f}<br>"
                f"max&nbsp;|F| err&nbsp;= {mag.max():.4f}<br>"
                f"mean cos sim&nbsp;= {cos.mean():.4f}<br>"
                f"min&nbsp;cos sim&nbsp;= {cos.min():.4f}<br>"
                f"ΔE − ⟨ΔE⟩ = {d_energy[frame_idx]:+.4f} eV<br>"
                f"atoms = {n_atoms}"
                f"</div>"
            ))

    slider = widgets.IntSlider(min=0, max=n_frames - 1, step=1, value=0, description="frame")
    play = widgets.Play(min=0, max=n_frames - 1, step=1, interval=int(1000 / PLAYBACK_FPS))
    widgets.jslink((play, "value"), (slider, "value"))
    slider.observe(lambda change: render_frame(change["new"]), names="value")

    render_frame(0)
    display(widgets.HBox([play, slider]))
    display(widgets.HBox([viewer_out, cb_mag_out, cb_dir_out, info_out]))
    display(err_out)


print("viewer ready")

## 7 · The gallery

Five trajectories, five viewers, predictions already cached by §5.

### GMD in-distribution split — monocarboxylic acids

#### O2C4H8

In [ ]:
show_trajectory("TestID", "O2C4H8_PBE.traj")

#### O2C10H20

In [ ]:
show_trajectory("TestID", "O2C10H20_PBE.traj")

### GMD out-of-distribution split — dicarboxylic acids

#### O4C4H6

In [ ]:
show_trajectory("TestOOD", "O4C4H6_PBE.traj")

#### O4C7H12

In [ ]:
show_trajectory("TestOOD", "O4C7H12_PBE.traj")

#### O4C10H18

In [ ]:
show_trajectory("TestOOD", "O4C10H18_PBE.traj")

## 8 · Summary, plus PET-SPICE and the other two notebooks

The interesting question for a family like UPET is whether picking the member trained on
the *right kind of matter* pays off: PET-SPICE is trained on SPICE (small organic
molecules), which is much closer to GMD's carboxylic acids than MAD's materials. Both are
loaded through the same factory call, so switching is one string.

In [ ]:
print(f"loading {COMPARE_MODEL} v{COMPARE_VERSION} ...")
compare_calc = CalculatorFactory.create(
    "upet", model=COMPARE_MODEL, version=COMPARE_VERSION, device=DEVICE
)
compare_rows = [
    evaluate_trajectory(split, molecule, calculator=compare_calc, tag=COMPARE_MODEL)
    for split, molecule in TRAJECTORIES
]
COMPARE_BASELINE = fit_element_baseline(compare_rows)
for r in compare_rows:
    r["energy_mae_per_atom"] = aligned_energy_mae_per_atom(r, COMPARE_BASELINE)

In [ ]:
header = (f"{'split':8s} {'molecule':10s} {'atoms':>5s} {'frames':>6s} "
          f"{'E MAE/atom*':>12s} {'F MAE':>9s} {'F max err':>9s} {'mean cos':>9s} {'min cos':>8s}")
for tag, table in ((MODEL, rows), (COMPARE_MODEL, compare_rows)):
    print(f"\n=== {tag} ===")
    print(header)
    print("-" * len(header))
    for r in table:
        max_component = max(
            float(np.abs(p - q).max()) for p, q in zip(r["pred_forces"], r["ref_forces"])
        )
        print(f"{r['split']:8s} {r['formula']:10s} {r['n_atoms']:5d} {r['n_frames']:6d} "
              f"{r['energy_mae_per_atom']:12.5f} {r['forces_mae']:9.5f} {max_component:9.4f} "
              f"{np.mean(r['cos_sim']):9.5f} {np.min(r['cos_sim']):8.4f}")
print("\n* baseline-aligned (§5). units: eV/atom, eV/Å, eV/Å")

### Force MAE across all four potentials

MACE-MP-0 and SIMURGH numbers are quoted from the sibling notebooks (measured 2026-08-05 on
these exact trajectories), not recomputed — SIMURGH's e3nn cannot be imported in this venv.

In [ ]:
REFERENCE_FORCES_MAE = {  # eV/Å, measured 2026-08-05 on the same 5 trajectories
    "MACE-MP-0 (medium)": {
        "C4H8O2": 0.14317, "C10H20O2": 0.13572,
        "C4H6O4": 0.17869, "C7H12O4": 0.15047, "C10H18O4": 0.14637,
    },
    "SIMURGH (trained on GMD)": {
        "C4H8O2": 0.01225, "C10H20O2": 0.01297,
        "C4H6O4": 1.31250, "C7H12O4": 0.22193, "C10H18O4": 0.51223,
    },
}

columns = [MODEL, COMPARE_MODEL, *REFERENCE_FORCES_MAE]
print(f"{'molecule':10s} {'split':8s} " + "".join(f"{c:>26s}" for c in columns))
print("-" * (19 + 26 * len(columns)))
for r, c in zip(rows, compare_rows):
    values = [
        r["forces_mae"],
        c["forces_mae"],
        *[REFERENCE_FORCES_MAE[name][r["formula"]] for name in REFERENCE_FORCES_MAE],
    ]
    print(f"{r['formula']:10s} {r['split']:8s} " + "".join(f"{v:26.5f}" for v in values))
print("\nzero-shot foundation models vs a model trained on this exact task (eV/Å)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for r, c in zip(rows, compare_rows):
    style = "-" if r["split"] == "TestID" else "--"
    line, = axes[0].plot(r["rel_mag"].mean(axis=1), style, linewidth=1.2,
                         label=f"{r['split']} {r['formula']}")
    axes[1].plot(c["rel_mag"].mean(axis=1), style, linewidth=1.2, color=line.get_color())
axes[0].set_title(f"{MODEL} — mean relative |F| error")
axes[1].set_title(f"{COMPARE_MODEL} — mean relative |F| error")
for ax in axes:
    ax.set_xlabel("frame")
    ax.grid(alpha=0.3)
shared = max(ax.get_ylim()[1] for ax in axes)
for ax in axes:
    ax.set_ylim(0, shared)  # same scale, so the two panels are directly comparable
axes[0].set_ylabel("mean relative |F| error")
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## 9 · Straight into MD / optimisation

`UPETCalculator` is already an ASE calculator, so there is nothing to wrap — unlike the
SIMURGH notebook (which needed a fragment-aware subclass) or the MACE one (which wrapped a
GOAL backbone in `GOALCalculator`). Under `goal.md` the same object comes from the YAML.

In [ ]:
from ase.optimize import BFGS

relaxed = ase.io.read(str(DATA_ROOT / "TestOOD" / "O4C10H18_PBE.traj"), index=0)
relaxed.calc = calc
e_start = relaxed.get_potential_energy()
f_start = np.abs(relaxed.get_forces()).max()

opt = BFGS(relaxed, logfile=None)
converged = opt.run(fmax=0.05, steps=25)

print(f"{relaxed.get_chemical_formula()} · {MODEL}-driven BFGS relaxation")
print(f"  start : E = {e_start:12.4f} eV   max|F| = {f_start:.4f} eV/Å")
print(f"  end   : E = {relaxed.get_potential_energy():12.4f} eV   "
      f"max|F| = {np.abs(relaxed.get_forces()).max():.4f} eV/Å")
print(f"  ΔE = {relaxed.get_potential_energy() - e_start:+.4f} eV over {opt.nsteps} steps "
      f"(converged: {converged})")

# Config-driven equivalent — configs/md/calculators/upet.yaml:
#   key: upet · model: pet-mad-s · version: "1.5.0" · variants: null · device: cuda

---

### What to take away

| | |
|---|---|
| Model | PET-MAD-1.5 small (`pet-mad-s` v1.5.0), r2SCAN, via `upet` 0.2.6 |
| Loaded via | `CalculatorFactory.create("upet", model=…, version=…, device=…)` |
| Package name | `upet` **is** the renamed `pet-mad` (`lab-cosmo/upet`); `pet-mad` 1.4.4 is the last old-name release |
| No ml-side backbone | unlike `mace_finetune`/`uma`, UPET has no `AtomicGraph` adapter — per-frame ASE calls, and fine-tuning runs through `goal-finetune-upet` (metatrain subprocess) |
| Energies | only meaningful after the per-element baseline fit of §5 |
| Kernel | `GOAL (mace-cuda, standalone)` |

**Environment notes** (all applied while writing this notebook):

* `upet` and its metatensor/metatrain stack were installed into `.venv-mace`.
* `nvalchemi-toolkit-ops` was **removed**: `metatomic_ase` prefers it for CUDA neighbour
  lists, but 0.4.0/0.4.1 crash with
  `TypeError: full(): argument 'size' … got float`. Without it, metatomic falls back to
  `vesin`, which works and still runs the model on the GPU. Restore with
  `uv pip install --python .venv-mace/bin/python nvalchemi-toolkit-ops` if a fixed release
  lands.
* GOAL's `UPETBuilder` needed three fixes before any of this ran: it hard-coded
  `variants={"energy": "energy"}` (upet reads that as a head named `energy/energy` and
  raises), it validated model ids against a `family × size` cross-product that both
  rejects released models like `pet-omad-l` and accepts non-existent ones like
  `pet-mad-xl`, and it could not forward `version`. See
  `tests/md/test_upet_calculator.py` for the regression tests.